# ADL 원시 데이터 적재 (CSV)

`sample/`의 CSV 3종(응급/평소/사망전, 각 5명 × 30일 = 150행, 총 450행)을 읽어 `adl_raw_records` 테이블에 **추가** 적재한다.

**기존 엑셀 노트북(`adl_raw_ingest.ipynb`)이 적재한 60건은 건드리지 않는다.** 이 노트북은 그 위에 CSV 450건을 더해 총 510건이 되도록 한다.

**source_type 매핑**
- `응급ADL_5명_30일.csv` → `"응급"`
- `평소ADL_5명_30일.csv` → `"평소"`
- `사망전ADL_5명_30일.csv` → `"사망전"`

**주의**: 이 노트북은 idempotent 하지 않다. 두 번 실행하면 CSV 데이터가 중복 적재된다.

In [1]:
import math
import os
import sys
from pathlib import Path

# 작업 디렉토리를 프로젝트 루트로 이동(.env.local·app import 해석용, 폴더 생성 아님).
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

SAMPLE_DIR = ROOT / "sample"
CSV_SOURCES = {
    "응급ADL_5명_30일.csv": "응급",
    "평소ADL_5명_30일.csv": "평소",
    "사망전ADL_5명_30일.csv": "사망전",
}

print(f"프로젝트 루트: {ROOT}")
for filename in CSV_SOURCES:
    print(f"  {filename}: 존재={(SAMPLE_DIR / filename).exists()}")

프로젝트 루트: C:\Users\kdwkd\documents\project\salpyeobom\salpyeobom-backend
  응급ADL_5명_30일.csv: 존재=True
  평소ADL_5명_30일.csv: 존재=True
  사망전ADL_5명_30일.csv: 존재=True


## 1. hex 디코딩 및 파서 (엑셀 노트북과 동일)

CSV의 5개 컬럼(`place_code_1_list`, `AIX_1_list`, `AIX_h_list`, `sleep_depth_1_list`, `outgoing_1_list`)은 **hex 문자열**이라 정수 리스트로 디코딩해야 한다.

- 1바이트: `place_code` / `sleep_depth` / `outgoing` (분 단위 1440개)
- 2바이트 big-endian: `AIX_1`(1440개) / `AIX_h`(시 단위 24개)
- `temp/humi/illu`는 `temp_00`~`temp_23` 개별 컬럼 → `extract_hourly` 가 수집

CSV 헤더에는 `death_date`, `death_record` 컬럼이 없으므로(사망전 파일에도 없음) 해당 필드는 자동으로 `None` 으로 들어간다.

In [2]:
def hex_to_int_list(value: object, width: int = 1) -> list[int] | None:
    """hex 문자열을 정수 리스트로 디코딩한다.

    width : 값 1개당 바이트 수. 1 = 1바이트, 2 = 2바이트 big-endian.
    hex 형식이 아니면 None 을 반환한다.
    """
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    s = str(value).strip().lower()
    if s.startswith("0x"):
        s = s[2:]
    if not s or len(s) % (2 * width) != 0:
        return None
    if any(c not in "0123456789abcdef" for c in s):
        return None
    raw = bytes.fromhex(s)
    if width == 1:
        return list(raw)
    return [int.from_bytes(raw[i:i + width], "big") for i in range(0, len(raw), width)]

In [3]:
def nan_to_none(v: object) -> object:
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    return v


def parse_date(v: object):
    """YYYYMMDD 정수 또는 날짜 문자열을 date 로 파싱."""
    if nan_to_none(v) is None:
        return None
    try:
        s = str(int(float(str(v))))
        if len(s) == 8:
            return pd.to_datetime(s, format="%Y%m%d").date()
    except (ValueError, OverflowError):
        pass
    parsed = pd.to_datetime(v, errors="coerce")
    return None if pd.isna(parsed) else parsed.date()


def parse_int(v: object) -> int | None:
    if nan_to_none(v) is None:
        return None
    try:
        return int(float(str(v)))
    except (ValueError, TypeError):
        return None


def parse_float(v: object) -> float | None:
    if nan_to_none(v) is None:
        return None
    try:
        return float(str(v))
    except (ValueError, TypeError):
        return None


def parse_str(v: object, max_len: int | None = None) -> str | None:
    if nan_to_none(v) is None:
        return None
    s = str(v).strip()
    if not s or s.lower() == "nan":
        return None
    return s[:max_len] if max_len else s


def parse_id(v: object, max_len: int) -> str:
    """ID 컬럼 전용 정규화. pandas가 NaN 섞인 컬럼을 float64로 읽어 '661.0'으로
    저장되는 문제를 막기 위해, 정수로 변환 가능하면 정수 문자열로 반환한다."""
    n = parse_int(v)
    if n is not None:
        return str(n)[:max_len]
    return parse_str(v, max_len) or ""


def extract_hourly(row: pd.Series, prefix: str) -> list | None:
    """temp_00 ~ temp_23 같은 시간별 컬럼 24개를 리스트로 모은다."""
    vals = [parse_float(row.get(f"{prefix}_{h:02d}")) for h in range(24)]
    return None if all(v is None for v in vals) else vals

In [4]:
def build_record(row: pd.Series, source_type: str) -> dict:
    """CSV 한 행을 AdlRawRecord 필드 dict 로 변환한다."""
    return {
        "source_type": source_type,
        # 기본 정보
        "care_recipient_id": parse_id(row.get("care_recipient_id"), 32),
        "age": parse_int(row.get("age")),
        "sex": parse_str(row.get("sex"), 1),
        "alone": parse_str(row.get("alone"), 1),
        "vision": parse_str(row.get("vision"), 16),
        "hearing": parse_str(row.get("hearing"), 16),
        "dosage": parse_str(row.get("dosage"), 16),
        "district": parse_str(row.get("district"), 64),
        "house_structure": parse_str(row.get("house_structure"), 16),
        "room_no": parse_int(row.get("room_no")),
        "bath_location": parse_str(row.get("bath_location"), 16),
        # 이벤트 정보 (CSV에 death_date/death_record 컬럼은 없음 → None)
        "lifeog_date": parse_date(row.get("lifeog_date")),
        "emergency_date": parse_date(row.get("emergency_date")),
        "emergency_record": parse_str(row.get("emergency_record")),
        "occurrence_place": parse_str(row.get("occurrence_place"), 32),
        "on_site": parse_str(row.get("on_site"), 16),
        "hospital_transfer": parse_str(row.get("hospital_transfer"), 16),
        "hospital_treatment": parse_str(row.get("hospital_treatment"), 16),
        "death_date": parse_date(row.get("death_date")),
        "death_record": parse_str(row.get("death_record")),
        # 분석 데이터 — hex 문자열을 정수 리스트로 디코딩
        "place_code_1_list": hex_to_int_list(row.get("place_code_1_list"), width=1),
        "aix_1_list": hex_to_int_list(row.get("AIX_1_list"), width=2),
        "aix_h_list": hex_to_int_list(row.get("AIX_h_list"), width=2),
        "aix_d": parse_float(row.get("AIX_d")),
        "aix_1_eq_0_repeat_count": parse_int(row.get("AIX_1_eq_0_repeat_count")),
        "total_aix_sum": parse_float(row.get("total_aix_sum")),
        "total_aix_inc_ratio": parse_float(row.get("total_aix_inc_ratio")),
        "night_aix_ratio": parse_float(row.get("night_aix_ratio")),
        "total_age_aix_ratio": parse_float(row.get("total_age_aix_ratio")),
        # 수면
        "sleep_depth_1_list": hex_to_int_list(row.get("sleep_depth_1_list"), width=1),
        "sleep_start_time_d": parse_str(row.get("sleep_start_time_d"), 8),
        "sleep_end_time_d": parse_str(row.get("sleep_end_time_d"), 8),
        "total_sleep_period": parse_float(row.get("total_sleep_period")),
        "total_sleep_aix_ratio": parse_float(row.get("total_sleep_aix_ratio")),
        # 목욕
        "bath_count_d": parse_int(row.get("bath_count_d")),
        "bath_time_d": parse_float(row.get("bath_time_d")),
        "bath_nomove_time": parse_float(row.get("bath_nomove_time")),
        "bath_count_in_sleep": parse_int(row.get("bath_count_in_sleep")),
        "bath_time_per_count": parse_float(row.get("bath_time_per_count")),
        "total_bath_average_count": parse_float(row.get("total_bath_average_count")),
        # 외출
        "outgoing_1_list": hex_to_int_list(row.get("outgoing_1_list"), width=1),
        "outgoing_count_d": parse_int(row.get("outgoing_count_d")),
        "outgoing_time_d": parse_float(row.get("outgoing_time_d")),
        "outgoing_late_night_count_d": parse_int(row.get("outgoing_late_night_count_d")),
        "outgoing_late_night_time_d": parse_float(row.get("outgoing_late_night_time_d")),
        "last_outgoing_time": parse_str(row.get("last_outgoing_time"), 16),
        "total_outgoing_average_time": parse_float(row.get("total_outgoing_average_time")),
        "total_outgoing_average_count": parse_float(row.get("total_outgoing_average_count")),
        # 시간별 환경 센서 — temp_00~temp_23 등 24개 컬럼 수집
        "temp_list": extract_hourly(row, "temp"),
        "humi_list": extract_hourly(row, "humi"),
        "illu_list": extract_hourly(row, "illu"),
    }

## 2. CSV 로드

CSV 헤더 첫 컬럼에 BOM(`\ufeff`)이 들어있어 `encoding="utf-8-sig"` 로 읽는다. 엑셀과 달리 CSV는 매 행에 `care_recipient_id` 값이 채워져 있어 forward fill 이 필요 없다.

In [5]:
frames = []
for filename, source_type in CSV_SOURCES.items():
    df = pd.read_csv(SAMPLE_DIR / filename, encoding="utf-8-sig")
    df["__source_type"] = source_type
    frames.append(df)
    print(f"[{source_type}] {len(df)}행 로드: {filename}")

records_df = pd.concat(frames, ignore_index=True)
print(f"총 {len(records_df)}행")
records_df[["care_recipient_id", "__source_type", "age", "lifeog_date"]].head()

[응급] 150행 로드: 응급ADL_5명_30일.csv
[평소] 150행 로드: 평소ADL_5명_30일.csv
[사망전] 150행 로드: 사망전ADL_5명_30일.csv
총 450행


C:\Users\kdwkd\AppData\Local\Temp\ipykernel_33396\3839052711.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["__source_type"] = source_type


,care_recipient_id,__source_type,age,lifeog_date
0,661,응급,78,20230101
1,661,응급,78,20230102
2,661,응급,78,20230103
3,661,응급,78,20230104
4,661,응급,78,20230105


## 3. DB 적재

Tortoise 초기화 → 기존 건수 확인 → bulk_create.

In [6]:
from tortoise import Tortoise

from app.database import TORTOISE_ORM
from app.models.adl_raw import AdlRawRecord

# 스키마는 aerich 가 관리하므로 generate_schemas() 는 호출하지 않는다.
if not Tortoise.apps:
    await Tortoise.init(config=TORTOISE_ORM)
print("DB 연결 완료")

DB 연결 완료


In [7]:
from tortoise.transactions import in_transaction

before = await AdlRawRecord.all().count()
print(f"적재 전 adl_raw_records: {before}건 (엑셀 노트북이 적재한 데이터)")

records = [
    AdlRawRecord(**build_record(row, row["__source_type"]))
    for _, row in records_df.iterrows()
]

async with in_transaction():
    await AdlRawRecord.bulk_create(records, batch_size=100)

print(f"CSV {len(records)}건 추가 적재 완료")

적재 전 adl_raw_records: 60건 (엑셀 노트북이 적재한 데이터)


CSV 450건 추가 적재 완료


## 4. 검증

- 총 건수: 기존 60 + CSV 450 = **510건**
- source_type 분포: 응급 180 (엑셀30 + CSV150) / 평소 150 / 사망전 150 / 사망 30 (엑셀만)
- 평소 샘플 1건의 배열 컬럼 길이가 모두 1440 또는 24 (또는 None)

In [8]:
total = await AdlRawRecord.all().count()
print(f"총 {total}건")

for st in ["응급", "평소", "사망전", "사망"]:
    cnt = await AdlRawRecord.filter(source_type=st).count()
    print(f"  source_type={st}: {cnt}건")

sample = await AdlRawRecord.filter(source_type="평소").first()
print(f"\n샘플(평소) care_recipient_id={sample.care_recipient_id}, lifeog_date={sample.lifeog_date}")
array_cols = [
    "place_code_1_list", "aix_1_list", "aix_h_list", "sleep_depth_1_list",
    "outgoing_1_list", "temp_list", "humi_list", "illu_list",
]
for col in array_cols:
    v = getattr(sample, col)
    length = len(v) if isinstance(v, list) else None
    head = v[:8] if isinstance(v, list) else v
    print(f"  {col}: 길이={length}, 앞부분={head}")

총 510건
  source_type=응급: 180건
  source_type=평소: 150건
  source_type=사망전: 150건
  source_type=사망: 30건

샘플(평소) care_recipient_id=661, lifeog_date=2023-01-01
  place_code_1_list: 길이=1440, 앞부분=[20, 20, 20, 20, 20, 20, 20, 20]
  aix_1_list: 길이=1440, 앞부분=[88, 288, 154, 104, 106, 135, 131, 169]
  aix_h_list: 길이=24, 앞부분=[71, 26, 26, 23, 27, 26, 21, 84]
  sleep_depth_1_list: 길이=1440, 앞부분=[0, 0, 0, 0, 0, 0, 0, 0]
  outgoing_1_list: 길이=1440, 앞부분=[254, 254, 254, 254, 254, 254, 254, 254]
  temp_list: 길이=24, 앞부분=[29.0, 28.3, 28.3, 28.5, 28.4, 28.4, 28.5, 28.9]
  humi_list: 길이=24, 앞부분=[62.0, 62.0, 62.1, 62.4, 61.6, 61.8, 62.3, 61.6]
  illu_list: 길이=24, 앞부분=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 4.0]


In [9]:
await Tortoise.close_connections()
print("DB 연결 종료")

DB 연결 종료
